# QUEST-HPV v3.1.3 workflow — Python 3.9 compatible

This notebook uses the **same Python interpreter as the active Jupyter kernel** for installation and all CLI calls. It stops immediately when a prerequisite file or output is missing, and architecture ablations are enabled explicitly in the primary workflow.

The primary workflow remains patient-disjoint and clone-first. HPV-negative tumour TIL clones are learned as explicit `nonNeo` controls by default, while the biological gate is reserved for final deployment probabilities. Checkpoint selection, calibration, and threshold selection use Validation only; the Independent Test remains locked.

In [1]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

import pandas as pd


def find_repository_root(start: Path) -> Path:
    """Find the nearest parent containing both pyproject.toml and src/quest_hpv."""
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "quest_hpv").is_dir():
            return candidate
    raise FileNotFoundError(
        "Cannot locate the QUEST-HPV repository root. Start Jupyter from the cloned "
        "repository or move this notebook into the repository/notebooks directory."
    )


ROOT = find_repository_root(Path.cwd())
os.chdir(ROOT)

print(f"Repository root : {ROOT}")
print(f"Kernel Python   : {sys.executable}")
print(f"Python version  : {sys.version.split()[0]}")
if sys.version_info < (3, 9):
    raise RuntimeError("QUEST-HPV v3.1.3 requires Python 3.9 or newer.")

Repository root : C:\Users\wenzh\Desktop\QUEST\QUEST-HPV-v3.1.3
Kernel Python   : C:\Users\wenzh\anaconda3\envs\deep_learning_TR\python.exe
Python version  : 3.9.21


## 0. Configure paths and execution switches

Edit only this cell when file names differ. The optional long-running analyses are disabled by default so **Run All** does not retrain multiple duplicate models.

In [3]:
# Input files
TRAIN = ROOT / "Model_construction_train_data.csv"
VAL = ROOT / "Model_construction_validation_data.csv"
TEST = ROOT / "Model_construction_test_data.csv"
META = ROOT / "Model_construction_patient_metadata.csv"
CONFIG = ROOT / "configs" / "recommended.json"
UNKNOWN = ROOT / "unknown_cells.csv"

# Primary outputs
RESULTS = ROOT / "QUEST_v3_1_results"
MODEL_BUNDLE = RESULTS / "quest_hpv_v3_1_ensemble.pt"
PREPROCESSOR = RESULTS / "preprocessor.joblib"
ARCHITECTURE_RESULTS = RESULTS / "architecture_ablation"
POSTHOC_RERUN_RESULTS = RESULTS / "interpretability_posthoc_rerun"
UNKNOWN_RESULTS = ROOT / "QUEST_v3_1_unknown_predictions"
UNKNOWN_CELL_LABEL_FILE = UNKNOWN_RESULTS / "unknown_cell_id_Label.csv"
UNKNOWN_CELL_DETAIL_FILE = UNKNOWN_RESULTS / "unknown_cell_predictions.csv"

# Main workflow switches
RUN_PRIMARY_TRAINING = True
RUN_UNKNOWN_PREDICTION = True
RUN_ARCHITECTURE_ABLATIONS = True  # retrained variants + CSV tables + publication figures
DEBUG_SINGLE_SEED = False          # True = faster smoke/debug run
SKIP_INTERPRETABILITY = False      # True = skip post-hoc modality/program/IG/occlusion analyses

# Optional standalone/development workflows
RUN_ONE_COMMAND_TRAIN_AND_PREDICT = False
RUN_DEVELOPMENT_SPLIT_SEARCH = False
RUN_MIXED_PATIENT_CV_PLAN = False
RUN_STANDALONE_ARCHITECTURE_ABLATIONS = False

In [5]:
def require_files(paths, stage: str) -> None:
    missing = [Path(path) for path in paths if not Path(path).is_file()]
    if missing:
        formatted = "\n".join(f"  - {path}" for path in missing)
        raise FileNotFoundError(f"{stage}: required file(s) not found:\n{formatted}")


def run_cli(*arguments) -> None:
    """Run QUEST-HPV with the current Jupyter kernel, independent of shell PATH."""
    command = [sys.executable, "-m", "quest_hpv", *map(str, arguments)]
    printable = subprocess.list2cmdline(command) if os.name == "nt" else shlex.join(command)
    print(f"$ {printable}", flush=True)
    subprocess.run(command, cwd=ROOT, check=True)


def add_optional_path(arguments, flag: str, path: Path):
    if path is not None and Path(path).is_file():
        arguments.extend([flag, str(path)])
    return arguments


require_files([ROOT / "pyproject.toml", CONFIG], "Repository preflight")

## 1. Install into the active Jupyter environment

The original failed because the package declared Python ≥3.10 while the active kernel was Python 3.9.21. Version 3.1.3 supports Python 3.9–3.12 and avoids Python-3.10-only type syntax.

In [7]:
install_command = [sys.executable, "-m", "pip", "install", "-e", str(ROOT)]
print("$ " + (subprocess.list2cmdline(install_command) if os.name == "nt" else shlex.join(install_command)))
subprocess.run(install_command, cwd=ROOT, check=True)

# Verify the same interpreter can import and invoke the package.
run_cli("--version")
run_cli("--help")

$ C:\Users\wenzh\anaconda3\envs\deep_learning_TR\python.exe -m pip install -e C:\Users\wenzh\Desktop\QUEST\QUEST-HPV-v3.1.3
$ C:\Users\wenzh\anaconda3\envs\deep_learning_TR\python.exe -m quest_hpv --version
$ C:\Users\wenzh\anaconda3\envs\deep_learning_TR\python.exe -m quest_hpv --help


## 2. Train the primary v3.1 model

The Independent Test remains patient-disjoint. Exact paired-CDR3 decontamination is label-blind, and calibration/threshold selection uses Validation only.

In [8]:
if RUN_PRIMARY_TRAINING:
    require_files([TRAIN, VAL, TEST, CONFIG], "Primary training preflight")
    train_args = [
        "train-evaluate",
        "--train-data", TRAIN,
        "--validation-data", VAL,
        "--test-data", TEST,
        "--config", CONFIG,
        "--output-dir", RESULTS,
    ]
    add_optional_path(train_args, "--metadata", META)
    train_args.append("--run-ablations" if RUN_ARCHITECTURE_ABLATIONS else "--skip-ablations")
    if DEBUG_SINGLE_SEED:
        train_args.append("--single-seed")
    if SKIP_INTERPRETABILITY:
        train_args.append("--skip-interpretability")
    run_cli(*train_args)

    required_outputs = [MODEL_BUNDLE, PREPROCESSOR, RESULTS / "ablation_manifest.json"]
    if RUN_ARCHITECTURE_ABLATIONS:
        required_outputs.extend([
            ARCHITECTURE_RESULTS / "architecture_ablation_metrics.csv",
            ARCHITECTURE_RESULTS / "architecture_ablation_paper_table.csv",
            ARCHITECTURE_RESULTS / "architecture_ablation_manifest.json",
        ])
    if not SKIP_INTERPRETABILITY:
        required_outputs.append(RESULTS / "interpretability" / "posthoc_ablation_manifest.json")
    require_files(required_outputs, "Primary training output check")

    print(f"Model bundle       : {MODEL_BUNDLE}")
    print(f"Preprocessor       : {PREPROCESSOR}")
    print(f"Ablation manifest  : {RESULTS / 'ablation_manifest.json'}")
    if RUN_ARCHITECTURE_ABLATIONS:
        display(pd.read_csv(ARCHITECTURE_RESULTS / "architecture_ablation_paper_table.csv"))
else:
    print("Skipped primary training (RUN_PRIMARY_TRAINING=False).")

$ C:\Users\wenzh\anaconda3\envs\deep_learning_TR\python.exe -m quest_hpv train-evaluate --train-data C:\Users\wenzh\Desktop\QUEST\QUEST-HPV-v3.1.3\Model_construction_train_data.csv --validation-data C:\Users\wenzh\Desktop\QUEST\QUEST-HPV-v3.1.3\Model_construction_validation_data.csv --test-data C:\Users\wenzh\Desktop\QUEST\QUEST-HPV-v3.1.3\Model_construction_test_data.csv --config C:\Users\wenzh\Desktop\QUEST\QUEST-HPV-v3.1.3\configs\recommended.json --output-dir C:\Users\wenzh\Desktop\QUEST\QUEST-HPV-v3.1.3\QUEST_v3_1_results --metadata C:\Users\wenzh\Desktop\QUEST\QUEST-HPV-v3.1.3\Model_construction_patient_metadata.csv --run-ablations
Model bundle       : C:\Users\wenzh\Desktop\QUEST\QUEST-HPV-v3.1.3\QUEST_v3_1_results\quest_hpv_v3_1_ensemble.pt
Preprocessor       : C:\Users\wenzh\Desktop\QUEST\QUEST-HPV-v3.1.3\QUEST_v3_1_results\preprocessor.joblib
Ablation manifest  : C:\Users\wenzh\Desktop\QUEST\QUEST-HPV-v3.1.3\QUEST_v3_1_results\ablation_manifest.json


,variant,partition,AP,normalized_AP,AUROC,precision_at_recall_0.20,precision_at_recall_0.50
0,QUEST-HPV v3.1 primary,Validation,0.484647,0.469568,0.948431,0.636364,0.596154
1,QUEST-HPV v3.1 primary,Independent_test,0.641216,0.637458,0.978189,1.000000,0.882353
2,RNA only,Validation,0.431161,0.414517,0.952283,0.677419,0.543860
3,RNA only,Independent_test,0.572299,0.567819,0.917726,1.000000,0.888889
4,TCR only,Validation,0.373918,0.355599,0.734689,0.777778,0.109929
5,TCR only,Independent_test,0.208473,0.200182,0.686860,0.666667,0.031746
6,VDJ only,Validation,0.524725,0.510819,0.917304,1.000000,0.407895
7,VDJ only,Independent_test,0.377113,0.370588,0.890445,0.875000,0.161616
8,RNA+TCR no VDJ,Validation,0.498568,0.483896,0.955670,0.647059,0.596154
9,RNA+TCR no VDJ,Independent_test,0.737067,0.734312,0.972300,1.000000,0.941176


## 3. Predict `unknown_cells.csv` with the locked ensemble

No preprocessing, V(D)J vocabulary, gene program, calibration temperature, or threshold is re-fitted on the unknown cohort. Each cell inherits its patient-clone prediction.

In [9]:
if RUN_UNKNOWN_PREDICTION:
    require_files([UNKNOWN, MODEL_BUNDLE, PREPROCESSOR], "Unknown-cohort prediction preflight")
    predict_args = [
        "predict",
        "--input-data", UNKNOWN,
        "--model-bundle", MODEL_BUNDLE,
        "--preprocessor", PREPROCESSOR,
        "--output-dir", UNKNOWN_RESULTS,
        "--eval-bag-views", 4,
        "--max-cells-per-clone", 128,
    ]
    run_cli(*predict_args)
    require_files(
        [UNKNOWN_CELL_LABEL_FILE, UNKNOWN_CELL_DETAIL_FILE],
        "Unknown-cohort prediction output check",
    )
else:
    print("Skipped unknown-cohort prediction (RUN_UNKNOWN_PREDICTION=False).")

$ C:\Users\wenzh\anaconda3\envs\deep_learning_TR\python.exe -m quest_hpv predict --input-data C:\Users\wenzh\Desktop\QUEST\QUEST-HPV-v3.1.3\unknown_cells.csv --model-bundle C:\Users\wenzh\Desktop\QUEST\QUEST-HPV-v3.1.3\QUEST_v3_1_results\quest_hpv_v3_1_ensemble.pt --preprocessor C:\Users\wenzh\Desktop\QUEST\QUEST-HPV-v3.1.3\QUEST_v3_1_results\preprocessor.joblib --output-dir C:\Users\wenzh\Desktop\QUEST\QUEST-HPV-v3.1.3\QUEST_v3_1_unknown_predictions --eval-bag-views 4 --max-cells-per-clone 128


In [10]:
import pandas as pd

if UNKNOWN_CELL_LABEL_FILE.is_file():
    unknown_cell_labels = pd.read_csv(UNKNOWN_CELL_LABEL_FILE)
    display(unknown_cell_labels.head(30))
else:
    print(
        f"No cell-level prediction table at {UNKNOWN_CELL_LABEL_FILE}. "
        "Run the preceding prediction cell after training, or set RUN_UNKNOWN_PREDICTION=True."
    )


,cell_id,Label
0,AAACCTGAGATCCCGC.1_17,nonNeo
1,AAACCTGAGGATGTAT.1_17,nonNeo
2,AAACCTGAGTCGCCGT.1_17,nonNeo
3,AAACCTGCACGCATCG.1_17,nonNeo
4,AAACCTGCACTTAAGC.1_17,nonNeo
5,AAACCTGCATCGATGT.1_17,nonNeo
6,AAACCTGGTCTGCGGT.1_17,nonNeo
7,AAACCTGGTTTCGCTC.1_17,nonNeo
8,AAACCTGTCTGCTGTC.1_17,nonNeo
9,AAACCTGTCTTATCTG.1_17,nonNeo


### Main unknown-cohort outputs

- `unknown_cell_id_Label.csv`: exactly one `cell_id,Label` row per input cell
- `unknown_cell_predictions.csv`: detailed cell-level probabilities, gate, uncertainty, and QC columns
- `unknown_RNA_cell_attention.csv`: multi-view averaged RNA attention (interpretability only)
- `unknown_VDJ_OOV.csv` and `unknown_input_audit.csv`: deployment QC
- `Figure_unknown_prediction_overview.{png,pdf,svg}`: cell-level ranking and uncertainty summary

The network remains clone-first internally, so cells belonging to the same patient-specific paired-TCR clone inherit the same prediction. No clone-level prediction CSV is written.


#### 4. Development-only split search

This reassigns only Train+Validation patients. The Independent Test remains locked and is never selected by performance.

In [11]:
SPLIT_RESULTS = ROOT / "QUEST_v3_1_split_search"
if RUN_DEVELOPMENT_SPLIT_SEARCH:
    require_files([TRAIN, VAL, TEST, CONFIG], "Development split-search preflight")
    args = [
        "optimize-development-split",
        "--train-data", TRAIN,
        "--validation-data", VAL,
        "--test-data", TEST,
        "--config", CONFIG,
        "--output-dir", SPLIT_RESULTS,
    ]
    add_optional_path(args, "--metadata", META)
    run_cli(*args)
    candidates = SPLIT_RESULTS / "development_split_candidates.csv"
    require_files([candidates], "Development split-search output check")
    display(pd.read_csv(candidates).head(20))
else:
    print("Optional development split search disabled.")

Optional development split search disabled.


##### 5. Mixed HPV-positive patient-held-out CV plan

Use this for development/model selection rather than repeatedly tuning on an inspected Independent Test.

In [12]:
CV_PLAN = ROOT / "mixed_patient_cv_plan_v3_1.csv"
if RUN_MIXED_PATIENT_CV_PLAN:
    require_files([META, CONFIG], "Mixed-patient CV preflight")
    run_cli(
        "make-mixed-patient-cv",
        "--metadata", META,
        "--config", CONFIG,
        "--output", CV_PLAN,
    )
    require_files([CV_PLAN], "Mixed-patient CV output check")
    display(pd.read_csv(CV_PLAN))
else:
    print("Optional mixed-patient CV planning disabled.")

Optional mixed-patient CV planning disabled.


## 6. Optional standalone retrained architecture ablation

Use this section when the primary model was already trained previously. `--primary-results-dir` reuses its `run_summary.json` and clone predictions, so the primary reference is not trained a second time. The other architectures are still retrained from scratch on identical patient partitions.

In [14]:
ABLATION_RESULTS = ROOT / "QUEST_v3_1_ablation"
if RUN_STANDALONE_ARCHITECTURE_ABLATIONS:
    require_files([TRAIN, VAL, TEST, CONFIG], "Architecture-ablation preflight")
    args = [
        "run-ablations",
        "--train-data", TRAIN,
        "--validation-data", VAL,
        "--test-data", TEST,
        "--config", CONFIG,
        "--output-dir", ABLATION_RESULTS,
    ]
    add_optional_path(args, "--metadata", META)
    if (RESULTS / "run_summary.json").is_file():
        args.extend(["--primary-results-dir", RESULTS])
    if DEBUG_SINGLE_SEED:
        args.append("--single-seed")
    run_cli(*args)
    ablation_file = ABLATION_RESULTS / "architecture_ablation_paper_table.csv"
    require_files([
        ablation_file,
        ABLATION_RESULTS / "architecture_ablation_manifest.json",
    ], "Architecture-ablation output check")
    display(pd.read_csv(ablation_file))
else:
    print("Standalone architecture ablation disabled; the primary workflow switch controls the integrated run.")

Standalone architecture ablation disabled; the primary workflow switch controls the integrated run.


###### 7. Locate and inspect all ablation outputs

In [15]:
manifest_path = RESULTS / "ablation_manifest.json"
if manifest_path.is_file():
    print(manifest_path.read_text(encoding="utf-8"))
else:
    print(f"Ablation manifest not found: {manifest_path}")

for table_path in [
    RESULTS / "interpretability" / "modality_ablation.csv",
    RESULTS / "interpretability" / "gene_program_ablation.csv",
    POSTHOC_RERUN_RESULTS / "modality_ablation.csv",
    POSTHOC_RERUN_RESULTS / "gene_program_ablation.csv",
    ARCHITECTURE_RESULTS / "architecture_ablation_metric_deltas.csv",
    ARCHITECTURE_RESULTS / "architecture_ablation_paired_bootstrap_summary.csv",
]:
    if table_path.is_file():
        print(f"\n{table_path}")
        display(pd.read_csv(table_path))

{
  "pipeline_version": "3.1.3",
  "posthoc_locked_model_ablation": {
    "analysis_type": "post_hoc_locked_model_perturbation",
    "evaluation_domain": "HPV_positive_independent_test",
    "temperature": 0.7918274402618408,
    "modality": {
      "status": "completed",
      "n_rows": 3,
      "table": "modality_ablation.csv",
      "figure_stem": "Figure_modality_ablation"
    },
    "gene_program": {
      "status": "completed",
      "n_rows": 8,
      "table": "gene_program_ablation.csv",
      "figure_stem": "Figure_gene_program_ablation"
    }
  },
  "retrained_architecture_ablation": {
    "status": "completed",
    "output_dir": "C:\\Users\\wenzh\\Desktop\\QUEST\\QUEST-HPV-v3.1.3\\QUEST_v3_1_results\\architecture_ablation",
    "n_metric_rows": 60,
    "reference_reused": true,
    "metrics_table": "C:\\Users\\wenzh\\Desktop\\QUEST\\QUEST-HPV-v3.1.3\\QUEST_v3_1_results\\architecture_ablation\\architecture_ablation_metrics.csv",
    "manifest": "C:\\Users\\wenzh\\Desktop\\QUE

,modality,n_HPV_positive_clones,baseline_AP_HPV_positive,ablated_AP_HPV_positive,delta_AP,mean_probability_change
0,RNA,2894,0.641216,0.334938,0.306278,-0.178216
1,VDJ,2894,0.641216,0.517431,0.123786,-0.000578
2,TCR,2894,0.641216,0.683372,-0.042156,-0.005852



C:\Users\wenzh\Desktop\QUEST\QUEST-HPV-v3.1.3\QUEST_v3_1_results\interpretability\gene_program_ablation.csv


,program,n_HPV_positive_clones,baseline_AP_HPV_positive,ablated_AP_HPV_positive,delta_AP,mean_probability_change
0,tcr_activation,2894,0.641216,0.639199,0.002017,1.268672e-05
1,exhaustion,2894,0.641216,0.641079,0.000138,-2.860878e-07
2,costimulation,2894,0.641216,0.641135,0.000081,1.088421e-07
3,tissue_residency,2894,0.641216,0.641185,0.000031,-5.605906e-08
4,stem_memory,2894,0.641216,0.641254,-0.000037,-3.356068e-06
5,ifng_response,2894,0.641216,0.641263,-0.000047,-6.472311e-07
6,proliferation,2894,0.641216,0.641370,-0.000154,5.284502e-07
7,cytotoxicity,2894,0.641216,0.641808,-0.000591,-5.190582e-06



C:\Users\wenzh\Desktop\QUEST\QUEST-HPV-v3.1.3\QUEST_v3_1_results\architecture_ablation\architecture_ablation_metric_deltas.csv


,variant,partition,domain,reference_variant,metric,reference_value,variant_value,delta_reference_minus_variant
0,RNA only,Independent_test,HPV_positive,QUEST-HPV v3.1 primary,n,2894.000000,2894.000000,0.000000
1,RNA only,Independent_test,HPV_positive,QUEST-HPV v3.1 primary,n_positive,30.000000,30.000000,0.000000
2,RNA only,Independent_test,HPV_positive,QUEST-HPV v3.1 primary,prevalence,0.010366,0.010366,0.000000
3,RNA only,Independent_test,HPV_positive,QUEST-HPV v3.1 primary,AP,0.641216,0.572299,0.068917
4,RNA only,Independent_test,HPV_positive,QUEST-HPV v3.1 primary,normalized_AP,0.637458,0.567819,0.069639
...,...,...,...,...,...,...,...,...
697,HPV free embedding control,Validation,all_patients,QUEST-HPV v3.1 primary,recall_at_precision_0.50,0.548387,0.629032,-0.080645
698,HPV free embedding control,Validation,all_patients,QUEST-HPV v3.1 primary,F1,0.310734,0.384880,-0.074145
699,HPV free embedding control,Validation,all_patients,QUEST-HPV v3.1 primary,balanced_accuracy,0.917366,0.932501,-0.015135
700,HPV free embedding control,Validation,all_patients,QUEST-HPV v3.1 primary,precision,0.188356,0.244541,-0.056185



C:\Users\wenzh\Desktop\QUEST\QUEST-HPV-v3.1.3\QUEST_v3_1_results\architecture_ablation\architecture_ablation_paired_bootstrap_summary.csv


,variant,partition,reference_variant,status,n_reference_clones,n_variant_clones,n_matched_clones,exact_clone_match,n_clones,n_patients,n_positive,observed_reference_AP,observed_variant_AP,observed_delta_AP,n_valid_bootstrap,bootstrap_mean_delta_AP,ci_low_95,ci_high_95,probability_reference_better,bootstrap_two_sided_p
0,RNA only,Validation,QUEST-HPV v3.1 primary,completed,4588,4588,4588,True,2181,3,62,0.484647,0.431161,0.053486,696,0.049399,0.041696,0.058904,1.000000,0.000000
1,RNA only,Independent_test,QUEST-HPV v3.1 primary,completed,4866,4866,4866,True,2894,2,30,0.641216,0.572299,0.068917,481,0.068917,0.068917,0.068917,1.000000,0.000000
2,TCR only,Validation,QUEST-HPV v3.1 primary,completed,4588,4588,4588,True,2181,3,62,0.484647,0.373918,0.110729,686,0.053385,-0.044064,0.127910,0.623907,0.752187
3,TCR only,Independent_test,QUEST-HPV v3.1 primary,completed,4866,4866,4866,True,2894,2,30,0.641216,0.208473,0.432743,520,0.432743,0.432743,0.432743,1.000000,0.000000
4,VDJ only,Validation,QUEST-HPV v3.1 primary,completed,4588,4588,4588,True,2181,3,62,0.484647,0.524725,-0.040078,683,-0.064458,-0.108348,-0.020501,0.000000,0.000000
5,VDJ only,Independent_test,QUEST-HPV v3.1 primary,completed,4866,4866,4866,True,2894,2,30,0.641216,0.377113,0.264103,486,0.264103,0.264103,0.264103,1.000000,0.000000
6,RNA+TCR no VDJ,Validation,QUEST-HPV v3.1 primary,completed,4588,4588,4588,True,2181,3,62,0.484647,0.498568,-0.013921,729,-0.011224,-0.019230,-0.005788,0.000000,0.000000
7,RNA+TCR no VDJ,Independent_test,QUEST-HPV v3.1 primary,completed,4866,4866,4866,True,2894,2,30,0.641216,0.737067,-0.095850,495,-0.095850,-0.095850,-0.095850,0.000000,0.000000
8,TCR+VDJ only,Validation,QUEST-HPV v3.1 primary,completed,4588,4588,4588,True,2181,3,62,0.484647,0.573805,-0.089158,705,-0.108036,-0.138144,-0.077432,0.000000,0.000000
9,TCR+VDJ only,Independent_test,QUEST-HPV v3.1 primary,completed,4866,4866,4866,True,2894,2,30,0.641216,0.350583,0.290634,499,0.290634,0.290634,0.290634,1.000000,0.000000
